# Spurious Background Reliance in Bird Classification

## Detecting and Mitigating Background Shortcuts with ERM and Group DRO on the Waterbirds Benchmark

---

**Course:** Machine Learning / Robustness

**Project:** Project 15 — Spurious Background Reliance in Bird Classification

**Date:** 2026

---

This notebook demonstrates that standard image classifiers learn spurious
background shortcuts, and that Distributionally Robust Optimization (Group DRO)
mitigates this failure mode on the canonical WILDS Waterbirds benchmark.

**Headline result:** Group DRO improves worst-group accuracy from **68.85%** (ERM)
to **85.51%** (+16.66 percentage points) while also improving average accuracy.

---


## 1. Problem Statement

Image classifiers often learn **spurious correlations** instead of the actual
class-defining features. The most common form is **background reliance**:
a model may classify birds based on the background (water vs land) rather
than the bird itself.

**Research questions:**

- Does standard training (ERM) exploit this background shortcut?
- How large is the gap between average and worst-subgroup accuracy?
- Can a robustness method (Group DRO) close this gap?
- Is the shortcut visible in the model's attention (saliency maps)?

**Why this matters:**
A model that relies on background shortcuts will fail systematically when
the "wrong" background appears (e.g., a waterbird photographed on land).
This is a real-world deployment hazard.

---


## 2. Dataset — Waterbirds (WILDS)

**Source:** Stanford WILDS benchmark (Sagawa et al., 2019; Koh et al., 2021)

**URL:** https://wilds.stanford.edu/datasets/

- **Total images:** 11,788
- **Train / Val / Test:** 4,795 / 1,199 / 5,794
- **Classes:** 2 (landbird, waterbird)
- **Subgroups:** 4 (bird type × background)

**The four subgroups and their training distribution:**

| Subgroup | Bird | Background | Train % | Role |
|---|---|---|---:|---|
| 0 | Landbird | Land | ~73% | **Majority** |
| 1 | Landbird | Water | ~4% | **Minority (HARD)** |
| 2 | Waterbird | Land | ~1% | **Minority (HARD)** |
| 3 | Waterbird | Water | ~22% | **Majority** |

**Key property:** The training set is **deliberately biased** — majority
subgroups make up ~95% of training data, while minority subgroups together
make up only ~5%. This creates a strong spurious correlation between
background and bird type.

---


## 3. Methodology

**Full workflow:**

1. **Data loading:** Use WILDS package to load Waterbirds from
   `kaggle/working/wilds_data/waterbirds_v1.0/` with image transforms
   (RandomResizedCrop + flip for train, Resize for eval).

2. **Split design:** Use WILDS official splits (train/val/test). Train
   uses heavily biased distribution; test has more balanced subgroups
   to make worst-group accuracy a meaningful metric.

3. **Feature extraction:** ImageNet-pretrained ResNet-50 backbone
   (last FC layer replaced with 2-class output).

4. **Modelling:**
   - **ERM baseline:** minimize average cross-entropy loss across all
     training examples (equal weight per example).
   - **Group DRO mitigation:** maintain per-group weights $q_g$ and
     upweight groups with higher loss via dual ascent.

5. **Validation:** Every epoch, compute both average and worst-group
   validation accuracy. **Select best checkpoint by worst-group val
   accuracy** (not average — selecting on average rewards shortcut learning).

6. **Analysis:** Per-subgroup test accuracy, gap analysis, and
   Grad-CAM saliency visualization.

---


## 4. Selected Models

### Baseline: ERM (Empirical Risk Minimization)

- **Objective:** $\min_\theta \frac{1}{N}\sum_i \ell(f_\theta(x_i), y_i)$
- Standard training procedure used in virtually all deep learning.
- Every example contributes equally to the gradient.
- On biased data, ERM minimizes average loss by exploiting the
  shortcut (high accuracy on majority subgroups, poor on minority).

### Main model: Group DRO (Distributionally Robust Optimization)

- **Objective:** $\min_\theta \max_{q \in \Delta} \sum_g q_g \cdot \mathbb{E}_g[\ell]$
- Maintains weights $q_g$ for each of the 4 subgroups.
- Updates weights via **dual ascent**: $q_g \leftarrow q_g \cdot \exp(\eta \cdot L_g)$
  then re-normalizes.
- Forces the optimizer to focus on the worst-performing subgroup
  (typically the rarest minority).

### Why these models?

- **ERM** is the natural baseline — it represents "do nothing special."
- **Group DRO** is the canonical WILDS mitigation, originally proposed
  by Sagawa et al. (2019). It uses group labels at training time
  (which Waterbirds provides via metadata).

### Architecture

Both models use the same backbone: **ResNet-50** pretrained on ImageNet
(23.5M parameters with our 2-class head). The only difference is the
loss function.

---


## 5. Evaluation Criterion

### Primary metric: **Worst-group accuracy**

Following the WILDS protocol, the headline metric is the **accuracy
on the worst-performing subgroup** (typically a minority subgroup):

$$\text{worst\_group\_acc} = \min_g \text{acc}(g)$$

### Why worst-group accuracy?

- **Average accuracy hides shortcut learning.** A model can get 90%
  average accuracy while completely failing on 30% of test examples.
- **Worst-group accuracy is a strict test.** It forces the model to
  perform well on ALL subgroups, not just the majority.
- **Real-world relevance.** Deployment failure on a minority subgroup
  is exactly the kind of bias we want to detect and prevent.

### Secondary metrics

- **Per-subgroup accuracy:** breakdown across all 4 subgroups.
- **Average accuracy:** standard metric for context.
- **Gap:** average − worst-group, the size of the disparity.
- **Grad-CAM saliency:** qualitative visual evidence of where the
  model "looks" — does it attend to the bird or the background?

### Why this protocol fits the problem

The Waterbirds dataset was specifically constructed to expose
background-reliance. Standard accuracy is insufficient because the
shortcut allows high average accuracy. The WILDS worst-group protocol
is the gold standard for evaluating this kind of robustness.

---


In [ ]:
# =============================================================
# Cell 1: Self-healing setup (data + bootstrap + verify)
# =============================================================

import os, subprocess, sys, time
from pathlib import Path

# --- 1. Install wilds ----------------------------------------------------
try:
    import wilds
    print("[setup] wilds already installed")
except ImportError:
    print("[setup] installing wilds...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "wilds"], check=True)

# --- 2. Verify environment ------------------------------------------------
import torch
print(f"[setup] PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[setup] GPU: {torch.cuda.get_device_name(0)}")

# --- 3. Download + extract dataset (if missing) -------------------------
DATA_ROOT = "/kaggle/working/wilds_data"
TARGET = os.path.join(DATA_ROOT, "waterbirds_v1.0")
metadata_csv = os.path.join(TARGET, "metadata.csv")

if os.path.exists(metadata_csv):
    print(f"[data] already exists at {TARGET}")
else:
    print(f"[data] not found, downloading...")
    os.makedirs(DATA_ROOT, exist_ok=True)
    TARBALL = "/kaggle/working/waterbird_complete95_forest2water2.tar.gz"
    URL = "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
    t0 = time.time()
    subprocess.run(["wget", "--progress=dot:giga", URL, "-O", TARBALL], check=True)
    print(f"[download] done in {(time.time()-t0)/60:.1f} min")
    subprocess.run(["tar", "-xzf", TARBALL, "-C", "/kaggle/working"], check=True)
    EXTRACTED = "/kaggle/working/waterbird_complete95_forest2water2"
    if os.path.exists(EXTRACTED):
        if os.path.exists(TARGET):
            subprocess.run(["rm", "-rf", TARGET], check=True)
        os.rename(EXTRACTED, TARGET)
    os.remove(TARBALL)
    print(f"[rename] moved to {TARGET}")

os.environ["WILDS_DATA_DIR"] = DATA_ROOT
sys.path.insert(0, "/kaggle/working")
print(f"[setup] WILDS_DATA_DIR = {DATA_ROOT}")


In [ ]:
# =============================================================
# Write all source files to erm/ package
# =============================================================

ERM = Path("/kaggle/working/erm")
ERM.mkdir(parents=True, exist_ok=True)
ERM.joinpath("__init__.py").write_text('ERM and Group DRO baselines for Waterbirds.\n')

# Write each source file
file_writes = {
'''
for name, path in src_files.items():
    file_writes += f'    "{name}": SRC_DIR / "{name}",
'

file_writes += '''}

for name, path in file_writes.items():
    ERM.joinpath(name).write_text(path.read_text())
    print(f"  wrote {name}")

# Verify compile
import py_compile
print("\nVerifying compilation...")
all_ok = True
for py_file in sorted(ERM.glob("*.py")):
    try:
        py_compile.compile(str(py_file), doraise=True)
        print(f"  [OK] {py_file.name}")
    except py_compile.PyCompileError as e:
        print(f"  [FAIL] {py_file.name}: {e}")
        all_ok = False

if not all_ok:
    raise SystemExit("Some files failed to compile")


In [ ]:
# =============================================================
# Verify model + inspect data
# =============================================================

for mod in list(sys.modules):
    if mod.startswith("erm"): del sys.modules[mod]

from erm.config import MODEL_NAME
from erm.model import build_model

model = build_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"[model] {MODEL_NAME}, {n_params:,} params")
with torch.no_grad():
    out = model(torch.randn(1, 3, 224, 224))
print(f"[model] output shape: {tuple(out.shape)}  (expect (1, 2))")

# Inspect data
from erm.inspect_data import main as inspect_main
inspect_main()


In [ ]:
# =============================================================
# Check if artifacts already exist (skip training if present)
# =============================================================

from pathlib import Path

erm_ckpt = Path("/kaggle/working/artifacts/models/erm_best.pt")
dro_ckpt = Path("/kaggle/working/artifacts/models/groupdro_best.pt")

if erm_ckpt.exists():
    print(f"ERM checkpoint:       EXISTS ({erm_ckpt.stat().st_size/1e6:.1f} MB)")
else:
    print(f"ERM checkpoint:       NOT FOUND")

if dro_ckpt.exists():
    print(f"Group DRO checkpoint: EXISTS ({dro_ckpt.stat().st_size/1e6:.1f} MB)")
else:
    print(f"Group DRO checkpoint: NOT FOUND")

if erm_ckpt.exists() and dro_ckpt.exists():
    print("\n[PIPELINE] Both checkpoints exist. Skipping training.")
    SKIP_TRAINING = True
else:
    print("\n[PIPELINE] Checkpoints missing. Training required (~80 min total).")
    SKIP_TRAINING = False


In [ ]:
# =============================================================
# Train ERM baseline (~40 min, or skip if artifacts exist)
# =============================================================

if not SKIP_TRAINING:
    for mod in list(sys.modules):
        if mod.startswith("erm"): del sys.modules[mod]
    from erm.train_erm import train_erm
    print("[train] Training ERM (50 epochs, ~40 min)...\n")
    info = train_erm(device="cuda")
    print(f"\n[done] ERM best worst-group val acc: {info['best_worst_group_val_acc']:.4f}")
else:
    print("[train] Skipping ERM training (using saved checkpoint)")


In [ ]:
# =============================================================
# Train Group DRO mitigation (~40 min, or skip if artifacts exist)
# =============================================================

if not SKIP_TRAINING:
    for mod in list(sys.modules):
        if mod.startswith("erm"): del sys.modules[mod]
    from erm.train_groupdro import train_groupdro
    print("[train] Training Group DRO (50 epochs, ~40 min)...\n")
    info = train_groupdro(device="cuda")
    print(f"\n[done] Group DRO best worst-group val acc: {info['best_worst_group_val_acc']:.4f}")
else:
    print("[train] Skipping Group DRO training (using saved checkpoint)")


In [ ]:
# =============================================================
# Evaluate both models on the test set
# =============================================================

for mod in list(sys.modules):
    if mod.startswith("erm"): del sys.modules[mod]
from pathlib import Path
from erm.evaluate import evaluate_on_test

print("=" * 70)
print("ERM - Test Set Evaluation")
print("=" * 70)
erm_ckpt = Path("/kaggle/working/artifacts/models/erm_best.pt")
erm_report = evaluate_on_test(erm_ckpt, device="cuda")

print("\n" + "=" * 70)
print("Group DRO - Test Set Evaluation")
print("=" * 70)
dro_ckpt = Path("/kaggle/working/artifacts/models/groupdro_best.pt")
dro_report = evaluate_on_test(dro_ckpt, device="cuda")


In [ ]:
# =============================================================
# Generate Grad-CAM saliency maps for both models
# =============================================================

for mod in list(sys.modules):
    if mod.startswith("erm"): del sys.modules[mod]
from pathlib import Path
from erm.saliency import generate_saliency_for_checkpoint

print("[saliency] Generating ERM heatmaps...")
generate_saliency_for_checkpoint(Path("/kaggle/working/artifacts/models/erm_best.pt"),
                                  device="cuda", per_group=2)
print("\n[saliency] Generating Group DRO heatmaps...")
generate_saliency_for_checkpoint(Path("/kaggle/working/artifacts/models/groupdro_best.pt"),
                                  device="cuda", per_group=2)

print("\n=== Artifacts produced ===")
for f in sorted(Path("/kaggle/working/artifacts").rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.relative_to('/kaggle/working')}  ({size_mb:.2f} MB)")


In [ ]:
# =============================================================
# Side-by-side comparison: ERM vs Group DRO
# =============================================================

import json
from pathlib import Path

erm_report = json.load(open("/kaggle/working/artifacts/results/erm_best_test_report.json"))
dro_report = json.load(open("/kaggle/working/artifacts/results/groupdro_best_test_report.json"))

print("=" * 70)
print("FINAL COMPARISON: ERM vs Group DRO")
print("=" * 70)

print(f"\n{'Method':<14s} {'Average':>10s} {'Worst-group':>13s} {'Gap':>8s}")
print("-" * 50)
for name, r in [("ERM", erm_report), ("Group DRO", dro_report)]:
    gap = r["overall_accuracy"] - r["worst_group_accuracy"]
    print(f"{name:<14s} {r['overall_accuracy']:>10.4f} {r['worst_group_accuracy']:>13.4f} {gap:>8.4f}")

print(f"\nPer-subgroup accuracy:")
print(f"  {'Group':<22s} {'ERM':>10s} {'Group DRO':>12s} {'Delta':>10s}")
print("  " + "-" * 56)
for gname in erm_report["per_group_accuracy"]:
    erm_acc = erm_report["per_group_accuracy"][gname]
    dro_acc = dro_report["per_group_accuracy"][gname]
    delta = dro_acc - erm_acc
    sign = "+" if delta > 0 else ""
    print(f"  {gname:<22s} {erm_acc:>10.4f} {dro_acc:>12.4f} {sign}{delta:>9.4f}")

improvement = dro_report["worst_group_accuracy"] - erm_report["worst_group_accuracy"]
if improvement > 0.10:
    verdict = "STRONG improvement in worst-group accuracy"
elif improvement > 0.05:
    verdict = "Moderate improvement in worst-group accuracy"
elif improvement > 0:
    verdict = "Small improvement in worst-group accuracy"
else:
    verdict = "No improvement (investigate)"

print(f"\nVerdict: {verdict}")
print(f"  Worst-group gain: {improvement*100:+.1f} percentage points")
print(f"  Average change:   {(dro_report['overall_accuracy']-erm_report['overall_accuracy'])*100:+.1f} percentage points")
print("=" * 70)


In [ ]:
# =============================================================
# Plot training curves (ERM vs Group DRO)
# =============================================================

import json
import matplotlib.pyplot as plt
from pathlib import Path

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for hist_path, label, color in [
    ("/kaggle/working/artifacts/models/erm_history.json", "ERM", "C0"),
    ("/kaggle/working/artifacts/models/groupdro_history.json", "Group DRO", "C1"),
]:
    if not Path(hist_path).exists():
        print(f"[warn] {hist_path} not found, skipping")
        continue
    hist = json.load(open(hist_path))
    epochs = [h["epoch"] for h in hist]
    avg = [h["val_avg_acc"] for h in hist]
    worst = [h["val_worst_group_acc"] for h in hist]
    axes[0].plot(epochs, avg, label=f"{label} avg", color=color, linewidth=2)
    axes[0].plot(epochs, worst, label=f"{label} worst-group", color=color,
                 linestyle="--", linewidth=2)
    axes[1].plot(epochs, worst, label=label, color=color, linewidth=2)

axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Validation Accuracy")
axes[0].set_title("ERM vs Group DRO - Validation Curves")
axes[0].legend(loc="lower right"); axes[0].grid(alpha=0.3)
axes[0].set_ylim(0.4, 1.0)

axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Worst-Group Accuracy")
axes[1].set_title("Worst-Group Accuracy Over Training")
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0.4, 1.0)

fig.tight_layout()
out = Path("/kaggle/working/artifacts/results/training_curves.png")
fig.savefig(out, dpi=120, bbox_inches="tight")
plt.show()
print(f"\n[plot] saved -> {out}")


## 6. Results

### 6.1 Headline Comparison

| Method | Average | Worst-group | Gap |
|---|---:|---:|---:|
| **ERM** | 86.61% | 68.85% | **17.77 pp** |
| **Group DRO** | 90.29% | 85.51% | **4.78 pp** |

**Key result:** Group DRO improves worst-group accuracy by **+16.66 percentage points**
(from 68.85% to 85.51%) while also improving average accuracy by **+3.68 points**.

### 6.2 Per-Subgroup Breakdown

| Subgroup | ERM | Group DRO | Delta |
|---|---:|---:|---:|
| landbird-on-land (majority) | 98.80% | 94.50% | -4.30 pp |
| landbird-on-water (minority) | 85.81% | 87.23% | +1.42 pp |
| **waterbird-on-land (minority)** | **68.85%** | **85.51%** | **+16.66 pp** |
| waterbird-on-water (majority) | 92.99% | 93.93% | +0.94 pp |

The biggest improvement is on **waterbird-on-land** (+16.66 pp), the subgroup where
ERM failed most. This subgroup has the **fewest training examples** (~56 images),
making it the hardest case for ERM but exactly what Group DRO targets.

### 6.3 Visual Evidence: Grad-CAM Saliency

The Grad-CAM heatmaps (saved to `artifacts/saliency/`) provide qualitative evidence:

- **ERM model**: attention is biased toward the **background** for minority
  subgroups (especially waterbird-on-land), confirming the model uses the
  background as a signal.
- **Group DRO model**: attention is more consistently centered on the **bird
  itself** across all subgroups, indicating it has learned actual bird features
  rather than background cues.

### 6.4 Training Dynamics

The training curves plot (saved above) shows:
- **ERM** quickly drives train loss to ~0.0001 (severe overfitting) but val
  worst-group accuracy plateaus around 0.5-0.7.
- **Group DRO** has higher train loss but achieves much better worst-group
  validation accuracy, demonstrating the trade-off works as intended.

---


## 7. Error Analysis and Limitations

### Where ERM still fails after Group DRO

After Group DRO, waterbird-on-land improves to 85.51%, but 14.49% of those
examples are still misclassified. Possible reasons:

- **Insufficient training examples**: waterbird-on-land has only ~56 training
  examples — even with upweighting, the model sees very few examples.
- **Strong visual similarity to landbirds on land**: some waterbird species
  (e.g., cormorants on rocks) genuinely look similar to landbirds when on
  land backgrounds.
- **Background still partially used**: even Group DRO may not fully eliminate
  background reliance when training data is so heavily biased.

### Limitations of this work

- **Single backbone**: Only ResNet-50 tested. Vision Transformers or larger
  ConvNets may behave differently.
- **Single dataset**: Waterbirds is one specific benchmark. Conclusions
  should be validated on other WILDS datasets (CelebA, CivilComments, etc.).
- **Single mitigation**: Only Group DRO compared. Other methods (IRM, CORAL,
  Just Train for Longer, reweighting) could provide additional baselines.
- **Group labels at training time**: Group DRO requires group labels during
  training, which may not be available in real-world settings.
- **Saliency resolution**: Grad-CAM produces 7x7 heatmaps from layer4.
  Higher-resolution methods (attention rollout for ViT) would give more
  detailed insights.

### What could be improved

- **Longer training**: 50 epochs may be insufficient; the model could
  benefit from more epochs with appropriate LR scheduling.
- **Data augmentation**: More aggressive augmentation (RandAugment,
  CutMix) might improve generalization on minority subgroups.
- **Ensemble methods**: Combining multiple models trained with different
  seeds could reduce variance.
- **Test-time intervention**: Methods like TENT or batchnorm adaptation
  could help at inference time.

---


## 8. Conclusion

This project demonstrated three key findings:

- **ERM exploits background shortcuts** in the Waterbirds dataset, producing
  a 17.77-percentage-point gap between average (86.61%) and worst-group
  (68.85%) accuracy.

- **Grad-CAM saliency maps visually confirm** the shortcut: ERM attends to
  the background (especially for minority subgroups) rather than the bird
  itself.

- **Group DRO closes this gap dramatically**, improving worst-group accuracy
  to 85.51% (a **+16.66 pp gain**) while also improving average accuracy to
  90.29%.

### Answer to the original research question

**Yes, ERM exploits background shortcuts, and yes, Group DRO effectively
mitigates this failure mode.**

The findings reproduce the central result of the WILDS benchmark paper
(Sagawa et al., 2019): standard training on biased data leads to shortcut
learning, and group-robust optimization is an effective mitigation.

### Practical implications

- Models deployed in real-world settings should be evaluated on **worst-group
  accuracy**, not just average accuracy.
- When training data is biased, consider using **group-robust optimization**
  methods like Group DRO.
- **Visualization** (Grad-CAM, attention maps) is essential for diagnosing
  whether models use the right features.

### Reproducibility

The complete pipeline is reproducible:
- Self-healing setup cell downloads data and writes source files.
- Two training cells (ERM + Group DRO) take ~40 min each on a single GPU.
- Evaluation and saliency cells take ~1 min each.
- All artifacts are saved to `/kaggle/working/artifacts/`.

---


## 9. References

1. **Sagawa, S., Koh, P. W., Hashimoto, T. B., & Liang, P.** (2020).
   *Distributionally Robust Neural Networks for Group Shifts*. ICLR 2020.
   [arXiv:1911.08731]

2. **Koh, P. W., Sagawa, S., et al.** (2021).
   *WILDS: A Benchmark of in-the-Wild Distribution Shifts*.
   ICML 2021. [arXiv:2012.07421]

3. **Selvaraju, R. R., et al.** (2017).
   *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based
   Localization*. ICCV 2017.

4. **He, K., Zhang, X., Ren, S., & Sun, J.** (2016).
   *Deep Residual Learning for Image Recognition*. CVPR 2016.

5. **Wah, C., Branson, S., Welinder, P., Perona, P., & Belongie, S.** (2011).
   *The Caltech-UCSD Birds-200-2011 Dataset*.
   California Institute of Technology.

---

## Appendix: How to run this notebook

**On Kaggle:**
1. Create a new notebook with GPU T4 x2 enabled
2. Paste all cells in order
3. Run cells 1-9 (setup + bootstrap + verify) - takes ~5 min
4. Run cells 10-12 (training, only if checkpoints don't exist) - takes ~80 min
5. Run cells 13-16 (evaluation, saliency, comparison) - takes ~2 min
6. Run cell 17 (training curves plot) - takes ~5 sec
7. View cells 18-20 (results, analysis, conclusion)

**Tip:** If you've already trained the models, just place the artifacts
in `/kaggle/working/artifacts/` and the notebook will skip training
automatically.

---
